# 02 — NLP Pipeline Experiments (Phase 2)

Verifies the Phase 2 processing modules (`cleaner`, `nlp_pipeline`, `categoriser`, `scorer`) built per `CLAUDE.md` Section 6.2 and Section 13's Phase 2 build order.

**Data available at time of writing:** Google Trends (372 rows, 5 tracked keywords) and, as of this update, 133 real news articles ingested via the NewsAPI ingester (`src/ingestion/news_scraper.py`, built per Section 7.3 — see Section 6 below for what changed vs. the spec). Reddit ingestion is still blocked by new-account restrictions.

Sections 1–5 below were written before real `raw_content` existed, so they run the text pipeline (`cleaner`/`nlp_pipeline`/`categoriser`) on **representative mock text** per `CLAUDE.md` instruction #8 (build with mock data when blocked on a source, and continue). They're left as-is — the mock-data findings still hold. **Section 6 is new**: it re-runs the same pipeline on the real ingested news articles and reports what actually happened, including two findings mock text couldn't have surfaced.

In [1]:
import os
import sys

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

from src.processing import cleaner, nlp_pipeline, categoriser, scorer
from src.storage import db

/Users/shloakshetty/Desktop/Fashion_Project/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Cleaner

In [2]:
raw = "<p>Loving the QUIET LUXURY aesthetic rn https://vogue.com/article café society vibes!! #streetwear</p>"
print(cleaner.clean_text(raw))

loving the quiet luxury aesthetic rn cafe society vibes streetwear


## 2. NLP pipeline (mock documents)

Three representative posts, styled like what the Reddit/News ingesters will eventually pull in from r/femalefashionadvice and fashion press.

In [3]:
MOCK_DOCUMENTS = [
    "Loving the quiet luxury aesthetic this season \u2014 oversized blazers, cargo pants, "
    "and chunky loafers are everywhere on r/femalefashionadvice right now.",
    "Barbiecore is officially dead, dark academia and mob wife aesthetic are the new streetwear "
    "staples for fall according to street style photographers at fashion week.",
    "Les blazers oversize sont partout cet automne, avec des pantalons larges et des mocassins.",
]

for doc in MOCK_DOCUMENTS:
    result = nlp_pipeline.process_document(doc)
    print(f"language={result['language']}")
    print(f"cleaned: {result['cleaned_text'][:80]}...")
    print(f"keywords: {[k['keyword'] for k in result['keywords']]}")
    print()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 36971.61it/s]

/Users/shloakshetty/Desktop/Fashion_Project/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:229: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/shloakshetty/Desktop/Fashion_Project/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:229: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/shloakshetty/Desktop/Fashion_Project/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:229: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/shloakshetty/Desktop/Fashion_Project/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:229: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/shloakshetty/Desktop/Fashion_Project/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:229: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/shloakshetty/Desktop/Fashion_Project/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:229: RuntimeWarning: invalid 

language=en
cleaned: loving the quiet luxury aesthetic this season oversized blazers cargo pants and ...
keywords: ['oversized blazers', 'loafers femalefashionadvice', 'cargo pants', 'luxury aesthetic', 'season oversized', 'quiet luxury', 'pants chunky', 'aesthetic season', 'blazers cargo', 'chunky loafers']

language=en
cleaned: barbiecore is officially dead dark academia and mob wife aesthetic are the new s...
keywords: ['barbiecore', 'barbiecore officially', 'academia mob', 'mob wife', 'dark academia', 'mob', 'dead dark', 'fashion', 'wife aesthetic', 'fashion week']

language=other
cleaned: les blazers oversize sont partout cet automne avec des pantalons larges et des m...
keywords: []



**Observation:** the third document (French) is correctly detected as non-English and skipped for keyword extraction — v1 is English-only per `CLAUDE.md`'s known limitations.

## 3. Categoriser: mapping extracted keywords to the taxonomy

In [4]:
sample_doc = MOCK_DOCUMENTS[0]
result = nlp_pipeline.process_document(sample_doc)

for kw in result["keywords"]:
    match = categoriser.categorise_keyword(kw["keyword"])
    print(f"{kw['keyword']:<30} -> {match['category']} ({match['match_type']})")

/Users/shloakshetty/Desktop/Fashion_Project/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:229: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/shloakshetty/Desktop/Fashion_Project/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:229: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/shloakshetty/Desktop/Fashion_Project/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:229: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 12749.40it/s]

oversized blazers              -> Silhouettes (semantic)
loafers femalefashionadvice    -> Clothing Items (semantic)
cargo pants                    -> Clothing Items (exact)
luxury aesthetic               -> Aesthetics (semantic)
season oversized               -> Silhouettes (semantic)
quiet luxury                   -> Aesthetics (exact)
pants chunky                   -> Clothing Items (semantic)
aesthetic season               -> None (emerging)
blazers cargo                  -> Clothing Items (semantic)
chunky loafers                 -> Clothing Items (semantic)


**Observation (known limitation, not a bug):** KeyBERT extracts multi-word phrases ("oversized blazers", "chunky loafers"), but the taxonomy in `CLAUDE.md` Section 5 lists single terms ("oversized", "loafers"). Whole-phrase exact/fuzzy matching doesn't catch a taxonomy term embedded inside a longer extracted phrase, so most compound phrases fall through to `emerging` even when a human would obviously categorise them. Single-word or already-canonical phrases ("cargo pants", "quiet luxury") match cleanly via exact match.

This is worth revisiting once real Reddit/News text is flowing — either by extracting unigrams alongside bigrams, or by checking whether any taxonomy term is a substring of the extracted phrase before falling back to fuzzy/semantic matching.

## 4. Scorer: momentum scoring on real Google Trends data

In [5]:
engine = db.init_db()
categoriser.seed_categories(engine)

signals = scorer.run()
for s in signals:
    print(s)

2026-09-09 18:12:41.032 | INFO     | src.storage.db:init_db:31 - Database initialised at sqlite:///data/fashion_trends.db


2026-09-09 18:12:41.033 | INFO     | src.processing.categoriser:seed_categories:72 - Seeded 6 taxonomy categories


2026-09-09 18:12:41.034 | INFO     | src.storage.db:init_db:31 - Database initialised at sqlite:///data/fashion_trends.db


2026-09-09 18:12:41.035 | INFO     | src.processing.scorer:score_keyword_from_google_trends:145 - Scored 'oversized': momentum=0.0434 status=stable


2026-09-09 18:12:41.037 | INFO     | src.processing.scorer:score_keyword_from_google_trends:145 - Scored 'cargo pants': momentum=0.0387 status=stable


2026-09-09 18:12:41.038 | INFO     | src.processing.scorer:score_keyword_from_google_trends:145 - Scored 'Y2K': momentum=-0.02 status=stable


2026-09-09 18:12:41.039 | INFO     | src.processing.scorer:score_keyword_from_google_trends:145 - Scored 'quiet luxury': momentum=2.5224 status=emerging


2026-09-09 18:12:41.039 | INFO     | src.processing.scorer:run:171 - Trend scoring complete: 4 signals computed


{'keyword': 'oversized', 'category_id': 2, 'date': '2026-08-31', 'mention_count': 65, 'momentum_score': 0.0434, 'source_diversity': 1, 'trend_status': 'stable'}
{'keyword': 'cargo pants', 'category_id': 1, 'date': '2026-08-31', 'mention_count': 69, 'momentum_score': 0.0387, 'source_diversity': 1, 'trend_status': 'stable'}
{'keyword': 'Y2K', 'category_id': 5, 'date': '2026-08-31', 'mention_count': 70, 'momentum_score': -0.02, 'source_diversity': 1, 'trend_status': 'stable'}
{'keyword': 'quiet luxury', 'category_id': 5, 'date': '2026-08-31', 'mention_count': 34, 'momentum_score': 2.5224, 'source_diversity': 1, 'trend_status': 'emerging'}


**Observation:** Google Trends' 0-100 interest scale sits well below the `mention_count` bands `classify_trend()` uses (100-1000 for "rising", >1000 for "peak"), which are tuned for raw document counts. So even a keyword with strong momentum (like `quiet luxury` at momentum > 2.5) can still land as `emerging` rather than `rising`/`peak` — these bands will start behaving as intended once Reddit/News mention counts (which run into the hundreds/thousands) are flowing into `trend_signals` alongside Google Trends.

Also note `source_diversity=1` for every signal right now — Google Trends is the only source ingested so far, so the diversity bonus in the momentum formula is inert until Reddit/News join in.

## 5. Real data: NewsAPI ingestion + NLP pipeline on real news articles

`src/ingestion/news_scraper.py` was built per `CLAUDE.md` Section 7.3, with one required deviation: the spec's `sources=vogue,elle,harpersbazaar,businessoffashion` filter fails outright — none of those are registered NewsAPI source IDs (confirmed via `client.get_sources()`: NewsAPI's ~125 sources skew toward general news/tech/business, with no fashion-specific publishers at all). Passing an invalid `sources` param errors the whole request, so the ingester queries by keyword only (the spec's 8-query `QUERIES` list), and deduplicates by URL across queries before inserting.

Live run: `python -m src.ingestion.news_scraper` fetched articles for all 8 queries, deduplicated 157 raw results down to 133, and inserted them into `raw_content`. Then `python -m src.processing.nlp_pipeline` processed all 133 through cleaning → language detection → keyword extraction → taxonomy categorisation, persisting 1,310 keyword rows.

In [6]:
from sqlalchemy import text

engine = db.init_db()

with engine.connect() as conn:
    row_count = conn.execute(text("SELECT COUNT(*) FROM raw_content WHERE source='news'")).scalar()
    print(f"news articles in raw_content: {row_count}")

    sample_titles = conn.execute(
        text("SELECT title FROM raw_content WHERE source='news' ORDER BY published_at DESC LIMIT 8")
    ).fetchall()
    print("\nMost recent headlines (unfiltered — includes off-topic noise):")
    for (title,) in sample_titles:
        print(f"  - {title[:100]}")

2026-09-09 18:12:41.042 | INFO     | src.storage.db:init_db:31 - Database initialised at sqlite:///data/fashion_trends.db


news articles in raw_content: 133

Most recent headlines (unfiltered — includes off-topic noise):
  - Adobe Stock Without Creative Cloud? Who Actually Wins on Price?
  - After June Runway Debut, Celine and Reebok’s Freestyle Lo Sneaker Is Here in 8 Colors
  - Quote of the Day by A$AP Rocky: “I'm here to break boundaries, man. That's all. I'm here to be the… 
  - ‘Musk’ Review: Alex Gibney’s Distressing Biography of an Uber-Wealthy Tech Bro Is Less Bombshell Tha
  - Which 2027 NFL Draft prospects performed the best in Week 1?
  - Sluggish offensive performance gives Auburn low 'SEC vibes'
  - Biotech filed for bankruptcy 11 weeks before its big FDA decision
  - Canva Is Quietly Publishing Nearly 1 Million Websites a Month


**Finding 1 — query noise:** single ambiguous keywords in the spec's `QUERIES` list (`"designer"`, `"runway"`) pull in a lot of off-topic NewsAPI results — NFL draft coverage, a biotech bankruptcy, a Musk documentary review, an airport runway story. Fashion trend intelligence built on keyword-only search (no `sources` allowlist, since none exist for fashion publishers) needs the downstream NLP/categorisation step to do real filtering work, not just enrichment — it's not optional polish.

In [7]:
total_keywords = nlp_pipeline.run(engine=engine)
print(f"keywords stored this run: {total_keywords}")

with engine.connect() as conn:
    distribution = conn.execute(
        text(
            """
            SELECT
                CASE WHEN keyword_type = 'emerging' THEN 'emerging (unmatched)' ELSE keyword_type END AS category,
                COUNT(*) AS n
            FROM keywords
            GROUP BY category
            ORDER BY n DESC
            """
        )
    ).fetchall()

print("\nKeyword category distribution across all 133 real articles:")
for category, n in distribution:
    print(f"  {category:<25} {n}")

/Users/shloakshetty/Desktop/Fashion_Project/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:229: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/shloakshetty/Desktop/Fashion_Project/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:229: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/shloakshetty/Desktop/Fashion_Project/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:229: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/shloakshetty/Desktop/Fashion_Project/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:229: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/shloakshetty/Desktop/Fashion_Project/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:229: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/shloakshetty/Desktop/Fashion_Project/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:229: RuntimeWarning: invalid 

/Users/shloakshetty/Desktop/Fashion_Project/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:229: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/shloakshetty/Desktop/Fashion_Project/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:229: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/shloakshetty/Desktop/Fashion_Project/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:229: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


/Users/shloakshetty/Desktop/Fashion_Project/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:229: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/shloakshetty/Desktop/Fashion_Project/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:229: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/shloakshetty/Desktop/Fashion_Project/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:229: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/shloakshetty/Desktop/Fashion_Project/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:229: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/shloakshetty/Desktop/Fashion_Project/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:229: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/shloakshetty/Desktop/Fashion_Project/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:229: RuntimeWarning: invalid 

/Users/shloakshetty/Desktop/Fashion_Project/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:229: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/shloakshetty/Desktop/Fashion_Project/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:229: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/shloakshetty/Desktop/Fashion_Project/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:229: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


/Users/shloakshetty/Desktop/Fashion_Project/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:229: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/shloakshetty/Desktop/Fashion_Project/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:229: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/shloakshetty/Desktop/Fashion_Project/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:229: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/shloakshetty/Desktop/Fashion_Project/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:229: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/shloakshetty/Desktop/Fashion_Project/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:229: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/shloakshetty/Desktop/Fashion_Project/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:229: RuntimeWarning: invalid 

...

/Users/shloakshetty/Desktop/Fashion_Project/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:229: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/shloakshetty/Desktop/Fashion_Project/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:229: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/shloakshetty/Desktop/Fashion_Project/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:229: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


/Users/shloakshetty/Desktop/Fashion_Project/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:229: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/shloakshetty/Desktop/Fashion_Project/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:229: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/shloakshetty/Desktop/Fashion_Project/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:229: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/shloakshetty/Desktop/Fashion_Project/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:229: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/shloakshetty/Desktop/Fashion_Project/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:229: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/shloakshetty/Desktop/Fashion_Project/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:229: RuntimeWarning: invalid 

/Users/shloakshetty/Desktop/Fashion_Project/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:229: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/shloakshetty/Desktop/Fashion_Project/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:229: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/shloakshetty/Desktop/Fashion_Project/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:229: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


2026-09-09 18:12:51.450 | INFO     | src.processing.nlp_pipeline:run:114 - NLP pipeline complete: 1310 keywords stored (58 matched taxonomy, 1252 emerging)


keywords stored this run: 1310

Keyword category distribution across all 133 real articles:
  emerging (unmatched)      1252
  Clothing Items            29
  Aesthetics                17
  Colours                   9
  Silhouettes               1
  Patterns & Textures       1
  Occasions                 1


**Finding 2 — a real bug, fixed, plus a real remaining gap.** The first pass of this pipeline matched only 6 of 1,310 extracted keywords (0.46%) to the taxonomy. Investigating why surfaced an actual bug in `categoriser.semantic_match()`: it compared each unmatched keyword against a **per-category blob embedding** — all ~50 keywords in a category joined into one string and embedded together. That dilutes the signal badly: `"handbag"` vs. the "Clothing Items" blob scored 0.28 (nowhere near the 0.6 threshold), even though `"handbag"` vs. the single term `"shoulder bag"` scores 0.75. Fixed by embedding every individual taxonomy term separately and taking the nearest neighbour instead of the category average — see `src/processing/categoriser.py`.

Re-running the pipeline after the fix: **58 of 1,310 keywords matched (4.4%)** — a ~10x improvement, and genuinely better (see the matched-keyword sample below: `"wardrobe"`, `"shoes"`, `"fast fashion"` now correctly match that previously wouldn't have). But 95.6% is still unmatched, which is too high to hand off to a dashboard without knowing why. Section 5b below investigates that directly, per-document rather than per-keyword.

In [8]:
with engine.connect() as conn:
    matched = conn.execute(
        text(
            """
            SELECT k.keyword, k.keyword_type, k.confidence, rc.title
            FROM keywords k
            JOIN raw_content rc ON k.content_id = rc.id
            WHERE k.keyword_type != 'emerging'
            ORDER BY k.confidence DESC
            """
        )
    ).fetchall()

for keyword, keyword_type, confidence, title in matched:
    print(f"{keyword:<15} {keyword_type:<16} conf={confidence:.2f}  {title[:80]}")

quiet luxury    Aesthetics       conf=1.00  How the Olsen Sisters Turned The Row Into a Quiet Luxury Powerhouse
bomber          Clothing Items   conf=1.00  Backfire to space: The Backfire bomber and Skif rocket system
sneaker         Clothing Items   conf=0.93  After June Runway Debut, Celine and Reebok’s Freestyle Lo Sneaker Is Here in 8 C
sneaker         Clothing Items   conf=0.93  Mfpen Brings New Balance’s 991 to the Dark Side of Luxury
camp            Clothing Items   conf=0.86  Cleo Camp Created the Reversible Tee Everybody Wants
bags            Clothing Items   conf=0.86  Fab Finds: Bags, Shoes, Bottoms, and More
backfire bomber Clothing Items   conf=0.71  Backfire to space: The Backfire bomber and Skif rocket system
sneaker colors  Clothing Items   conf=0.68  After June Runway Debut, Celine and Reebok’s Freestyle Lo Sneaker Is Here in 8 C
sacred white    Colours          conf=0.62  Are white bucks sacred?
york fashion    Aesthetics       conf=0.61  Mayor Mamdani, Can You Fix NY

**Finding 3 — a real false positive:** `"bomber"` matched `Clothing Items` (confidence 1.0, exact match — the taxonomy lists "bomber" as a jacket style) on the article *"Backfire to space: The Backfire bomber and Skif rocket system"* — a Russian military aircraft article, not fashion. Single-word taxonomy terms that are ambiguous outside a fashion context (`bomber` = jacket or aircraft; `camp` = clothing style or campsite) will misfire without document-level topical context. `"quiet luxury"` and the Reebok/sneaker matches, by contrast, are genuinely correct — multi-word or fashion-specific terms don't have this problem.

This is exactly the kind of false positive that source-diversity and sentiment scoring (not yet built) would help catch — a single mismatched keyword from one ambiguous article is a weak signal on its own, and the momentum formula's `source_diversity` bonus is designed to reward the same term appearing across independent sources, not one fluke match.

## 5b. Why is 95.6% still unmatched? Per-document root cause

Finding 2 fixed a real bug and got a real 10x improvement, but the unmatched rate is still high. Per-keyword stats can't distinguish "the matcher is bad" from "most of this content isn't fashion" — so this checks per-*document*: how many of the 133 articles have at least one taxonomy-matched keyword at all?

In [9]:
with engine.connect() as conn:
    docs = conn.execute(
        text(
            """
            SELECT rc.id, rc.title,
                   SUM(CASE WHEN k.keyword_type != 'emerging' THEN 1 ELSE 0 END) AS matched
            FROM raw_content rc
            LEFT JOIN keywords k ON k.content_id = rc.id
            WHERE rc.source = 'news'
            GROUP BY rc.id
            """
        )
    ).fetchall()

with_match = [d for d in docs if d[2] > 0]
without_match = [d for d in docs if d[2] == 0]
print(f"articles with >=1 taxonomy-matched keyword: {len(with_match)}/{len(docs)}")
print(f"articles with zero taxonomy-matched keywords: {len(without_match)}/{len(docs)}")

print("\nSample of zero-match article titles:")
import random
random.seed(1)
for title, _ in random.sample([(d[1], d[2]) for d in without_match], 12):
    print(f"  - {title[:85]}")

articles with >=1 taxonomy-matched keyword: 27/133
articles with zero taxonomy-matched keywords: 106/133

Sample of zero-match article titles:
  - A Wave of New York Office Tower Listings Tests the Market’s Recovery
  - A royal rider! Hard-working Princess Anne attends Burghley Horse Trials - having won 
  - The big Gen Z market taking over how India spends, saves and strategises
  - side-dog added to PyPI
  - Quote of the Day by Billie Eilish: “I’m not going to say I’m cool, because… – Inspiri
  - Bay State Ballers: How Massachusetts Ties Connect Miami’s Most Underrated X-Factors i
  - Teri Hatcher will star as Miranda Priestley in the West End’s ‘The Devil Wears Prada’
  - Acer Reports Revenues for August at NT$30.18 Billion and Year-to-August at NT$214.81 
  - Books that foster a sense of warmth and good conversation
  - F1’s newest race expected to be an ‘incredible’ thrill ride
  - Sensex today | Stock Market Highlights: Sensex down 555 pts, Nifty ends at 23,635 as 
  - EXCLUSIVE:

**Root cause, confirmed:** only 27/133 articles (20%) have any taxonomy-matched keyword at all. The zero-match sample above is decisive — real estate listings, PyPI package releases, a Bake Off recap, F1 racing, Sensex stock market coverage, aluminum market forecasts, trade tariffs. These aren't fashion articles that the matcher failed on; they're **not fashion content**, correctly landing as unmatched. This is Finding 1's query noise (ambiguous single-word queries like `"designer"` and `"runway"`), now measured precisely: it accounts for the large majority of the unmatched rate, not the matching logic.

There's a smaller secondary cause worth naming too — the cell below shows one genuinely fashion-relevant article that still got zero matches, and why.

In [10]:
with engine.connect() as conn:
    row = conn.execute(
        text("SELECT id, title FROM raw_content WHERE title LIKE '%Duolingo Creates Capsule%'")
    ).fetchone()
    kws = conn.execute(
        text("SELECT keyword, keyword_type FROM keywords WHERE content_id = :id"), {"id": row[0]}
    ).fetchall()

print(f"Article: {row[1]}")
print("Extracted keywords (all emerging):")
for kw, kw_type in kws:
    print(f"  - {kw}")

Article: EXCLUSIVE: Duolingo Creates Capsule Collection and Hosts Pop-Up During NYFW
Extracted keywords (all emerging):
  - nyfw duolingo
  - duolingo
  - duolingo debuts
  - duolingo creates
  - exclusive duolingo
  - fashion capsule
  - phonetic fashion
  - nyfw pop
  - debuts phonetic
  - pop nyfw


**Finding 4 — secondary cause: brand-name dominance + genuine taxonomy coverage gaps.** This article is unambiguously fashion news (a NYFW capsule collection launch) but every extracted keyword is `emerging`. Two reasons, both structural rather than bugs:

1. The brand name `"Duolingo"` dominates the title and extraction, and brand names are deliberately *not* in the Section 5 taxonomy — they belong to NER + the `brands` table (`CLAUDE.md` Section 6.2 Step 4), which is explicitly deferred (Phase 2 build order: "keyword extraction only first").
2. The one genuinely fashion-relevant phrase extracted, `"fashion capsule"`, has no match — `"capsule collection"` isn't in the Section 5 taxonomy at all. This is an honest taxonomy gap: Section 5 covers items/silhouettes/colours/patterns/aesthetics/occasions, but has no category for fashion industry events or product-line concepts (`"capsule collection"`, `"NYFW"`, `"trunk show"`, etc.).

Neither of these is fixable by tuning the matcher — they need brand NER (already on the roadmap) and a taxonomy extension (not currently planned), respectively. Worth flagging for prioritisation, not silently working around.

## 6. Summary: what's real vs. mock in this notebook

| Component | Status |
|---|---|
| `cleaner.clean_text` | Real logic, demoed on synthetic text (Section 1) and run on all 133 real articles (Section 5) |
| `nlp_pipeline` (language detect + keyword extraction) | Validated on mock text (Section 2) **and real data**: 133 real news articles, 1,310 keywords extracted (Section 5) |
| `categoriser` (exact/fuzzy/semantic taxonomy matching) | Validated on mock keywords (Section 3) **and real data** — a real bug found and fixed (Finding 2), a real false positive found (Finding 3), root cause of the remaining gap quantified (Section 5b) |
| `scorer` (momentum + classification) | **Real data** — 372 real Google Trends rows, real trend_signals written to the DB (Section 4) |
| `news_scraper` (NewsAPI ingester) | **Real data** — 133 articles ingested live, one required deviation from spec (no fashion-specific NewsAPI sources exist) documented in Section 5 |

**What real data changed vs. the mock-text predictions:** Sections 2–3 (mock text) correctly predicted the *mechanism* of the compound-phrase matching gap, but real data was needed to find the actual dominant cause. Investigating the unmatched rate surfaced a genuine bug (`semantic_match()` diluting its signal with category-blob embeddings — fixed, ~10x improvement) and then, at the per-document level, showed that the bug was never the main story: 80% of ingested articles simply aren't fashion content, a direct consequence of NewsAPI's keyword-only search (no fashion-specific `sources` exist to filter by). The remaining fashion-relevant misses trace to two structural gaps — brand names correctly deferred to NER, and real taxonomy coverage holes (e.g. no "capsule collection"/NYFW concept) — not to anything fixable by tuning the matcher further.

**Recommendation before Phase 3:** the query-noise problem (80% of articles) is the highest-leverage fix available and isn't a matching-logic problem — it needs either better `QUERIES` (fewer ambiguous single words like `"designer"`/`"runway"`) or a post-fetch relevance filter before ingestion. Worth a decision before spending more effort on `trend_signals` conclusions built on this data. Reddit remains blocked by account restrictions; revisit when available.